In [3]:
import pandas as pd
import numpy as np
import os

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR = ""
INFLOWS_PATH = f"{BASE_DIR}/data/raw/unhcr_inflows_raw.csv"
OUTFLOWS_PATH = f"{BASE_DIR}/data/raw/unhcr_outflows_raw.csv"
INTERIM_PATH = f"{BASE_DIR}/data/interim/"
os.makedirs(INTERIM_PATH, exist_ok=True)

# ── Country name standardization ───────────────────────────────────────────
# UNHCR uses variant names — standardize to match ISO crosswalk
COUNTRY_NAME_MAP = {
    "Dominican Rep.": "Dominican Republic",
    "Venezuela (Bolivarian Republic of)": "Venezuela",
    "Trinidad and Tobago": "Trinidad and Tobago",
    "Bahamas": "Bahamas",
}

# ISO3 lookup for your countries
ISO3_LOOKUP = {
    "Barbados": "BRB", "Belize": "BLZ", "Bahamas": "BHS",
    "Colombia": "COL", "Costa Rica": "CRI", "Cuba": "CUB",
    "Dominican Republic": "DOM", "El Salvador": "SLV",
    "Guatemala": "GTM", "Guyana": "GUY", "Haiti": "HTI",
    "Honduras": "HND", "Jamaica": "JAM", "Nicaragua": "NIC",
    "Panama": "PAN", "Puerto Rico": "PRI", "Suriname": "SUR",
    "Trinidad and Tobago": "TTO", "Venezuela": "VEN",
}

COUNTRY_ROLE = {
    "COL": "pressure_input", "VEN": "pressure_input",
}

# Your full country set
TIER1_ISOS = {
    "BLZ", "CRI", "CUB", "DOM", "SLV", "GTM", "GUY",
    "HTI", "HND", "JAM", "NIC", "PAN", "PRI", "SUR",
    "TTO", "BHS", "BRB", "COL", "VEN",
}

# Months to expand annual data into
MONTHS = list(range(1, 13))


def load_raw(path: str, label: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    print(f"  {label}: {df.shape[0]} rows × {df.shape[1]} columns")
    return df


def standardize_names(df: pd.DataFrame,
                      country_col: str) -> pd.DataFrame:
    """Standardize country names in a given column."""
    df[country_col] = df[country_col].replace(COUNTRY_NAME_MAP)
    return df


def add_iso3(df: pd.DataFrame, country_col: str,
             iso_col_existing: str, new_col: str) -> pd.DataFrame:
    """
    Add clean ISO3 column.
    Prefer existing ISO column from UNHCR, fall back to lookup by name.
    """
    # Use existing ISO column, clean it up
    df[new_col] = df[iso_col_existing].str.strip().str.upper()

    # Override with lookup for known problem cases
    name_based = df[country_col].map(ISO3_LOOKUP)
    df[new_col] = df[new_col].where(df[new_col].notna(), name_based)

    return df


def clean_inflows(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean inflows file — people arriving INTO your region.
    Focus column: Country of Asylum = your Tier 1 countries.

    Key columns:
      - Refugees            : recognized refugee status
      - Asylum-seekers      : pending determination
      - total_displaced_in  : sum of above two
    """
    df = standardize_names(df, "Country of Asylum")
    df = standardize_names(df, "Country of Origin")
    df = add_iso3(df, "Country of Asylum", "Country of Asylum ISO", "asylum_iso3")
    df = add_iso3(df, "Country of Origin", "Country of Origin ISO", "origin_iso3")

    # Filter to your countries as asylum countries
    df = df[df["asylum_iso3"].isin(TIER1_ISOS)].copy()

    # Build total displaced inflow
    df["total_displaced_in"] = df["Refugees"] + df["Asylum-seekers"]

    # Select and rename
    df = df[[
        "Year", "asylum_iso3", "origin_iso3",
        "Country of Asylum", "Country of Origin",
        "Refugees", "Asylum-seekers",
        "total_displaced_in",
        "IDPs", "Stateless persons",
    ]].rename(columns={
        "Year": "year",
        "Country of Asylum": "country_asylum",
        "Country of Origin": "country_origin",
        "Refugees": "refugees_in",
        "Asylum-seekers": "asylum_seekers_in",
        "IDPs": "idps",
        "Stateless persons": "stateless",
    })

    print(f"  Inflows after filter: {len(df)} rows")
    print(f"  Asylum countries: {df['asylum_iso3'].nunique()}")
    return df


def clean_outflows(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean outflows file — people fleeing FROM your region.
    Focus column: Country of Origin = your Tier 1 countries.

    Key columns:
      - Refugees            : recognized refugee status
      - Asylum-seekers      : pending determination
      - total_displaced_out : sum of above two
    """
    df = standardize_names(df, "Country of Asylum")
    df = standardize_names(df, "Country of Origin")
    df = add_iso3(df, "Country of Asylum", "Country of Asylum ISO", "asylum_iso3")
    df = add_iso3(df, "Country of Origin", "Country of Origin ISO", "origin_iso3")

    # Filter to your countries as origin countries
    df = df[df["origin_iso3"].isin(TIER1_ISOS)].copy()

    # Build total displaced outflow
    df["total_displaced_out"] = df["Refugees"] + df["Asylum-seekers"]

    # Select and rename
    df = df[[
        "Year", "origin_iso3", "asylum_iso3",
        "Country of Origin", "Country of Asylum",
        "Refugees", "Asylum-seekers",
        "total_displaced_out",
    ]].rename(columns={
        "Year": "year",
        "Country of Origin": "country_origin",
        "Country of Asylum": "country_asylum",
        "Refugees": "refugees_out",
        "Asylum-seekers": "asylum_seekers_out",
    })

    print(f"  Outflows after filter: {len(df)} rows")
    print(f"  Origin countries: {df['origin_iso3'].nunique()}")
    return df


def aggregate_annual_stocks(df_in: pd.DataFrame,
                             df_out: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate to country-year level, then expand to country-month.

    Inflows aggregated by asylum country (total people hosted).
    Outflows aggregated by origin country (total people displaced abroad).

    Annual→monthly expansion:
      monthly_baseline = annual_stock / 12
      This is Priority 2 — will be overridden by situation monthly data
      where available (Priority 1, added in later script).
    """
    # Aggregate inflows by asylum country + year
    inflow_agg = df_in.groupby(["year", "asylum_iso3"]).agg(
        total_refugees_in=("refugees_in", "sum"),
        total_asylum_seekers_in=("asylum_seekers_in", "sum"),
        total_displaced_in=("total_displaced_in", "sum"),
        total_idps=("idps", "sum"),
        n_origin_countries=("origin_iso3", "nunique"),
    ).reset_index().rename(columns={"asylum_iso3": "ISO3"})

    # Aggregate outflows by origin country + year
    outflow_agg = df_out.groupby(["year", "origin_iso3"]).agg(
        total_refugees_out=("refugees_out", "sum"),
        total_asylum_seekers_out=("asylum_seekers_out", "sum"),
        total_displaced_out=("total_displaced_out", "sum"),
        n_destination_countries=("asylum_iso3", "nunique"),
    ).reset_index().rename(columns={"origin_iso3": "ISO3"})

    # Merge inflows and outflows on country + year
    combined = pd.merge(
        inflow_agg, outflow_agg,
        on=["year", "ISO3"], how="outer"
    ).fillna(0)

    # Net displacement pressure = inflow + outflow
    # Both directions create resource demand on the country
    combined["net_displacement_pressure"] = (
        combined["total_displaced_in"] + combined["total_displaced_out"]
    )

    return combined


def expand_to_country_month(df_annual: pd.DataFrame) -> pd.DataFrame:
    """
    Expand annual country stocks to country-month grid.

    Method: divide annual stock evenly across 12 months.
    This is the Priority 2 baseline — flat distribution assumes
    displacement stock is relatively stable within a year.
    Disaster shocks (EM-DAT) will be layered on in the master build.

    Source flag marks these as annual proxy — will be overridden
    by Priority 1 situation data where available.
    """
    records = []

    for _, row in df_annual.iterrows():
        for month in MONTHS:
            record = row.to_dict()
            record["month"] = month
            record["year_month"] = (
                f"{int(row['year'])}-{str(month).zfill(2)}"
            )
            # Monthly baseline = annual / 12
            record["monthly_displaced_in_baseline"] = round(
                row["total_displaced_in"] / 12, 2
            )
            record["monthly_displaced_out_baseline"] = round(
                row["total_displaced_out"] / 12, 2
            )
            record["monthly_pressure_baseline"] = round(
                row["net_displacement_pressure"] / 12, 2
            )
            record["displacement_source"] = "unhcr_annual_proxy"
            records.append(record)

    df_cm = pd.DataFrame(records)

    # Add country role
    df_cm["country_role"] = df_cm["ISO3"].map(COUNTRY_ROLE).fillna("tier1")

    # Sort
    df_cm = df_cm.sort_values(["ISO3", "year", "month"]).reset_index(drop=True)

    return df_cm


def validate(df_in: pd.DataFrame, df_out: pd.DataFrame,
             df_cm: pd.DataFrame):
    print("\n── Validation ──────────────────────────────")

    # Year range
    assert df_cm["year"].between(2019, 2024).all()
    print("  ✅ Year range: 2019–2024")

    # Each country should have 72 months (6 years × 12)
    month_counts = df_cm.groupby("ISO3")["month"].count()
    expected = 72
    off = month_counts[month_counts != expected]
    if len(off) > 0:
        print(f"  ⚠️  Countries not at 72 months: {off.to_dict()}")
    else:
        print("  ✅ All countries have 72 month records")

    # Check Venezuela and Nicaragua are top inflow sources
    top_in = df_in.groupby("country_origin")["total_displaced_in"].sum().nlargest(3)
    print(f"  ✅ Top inflow origins: {top_in.index.tolist()}")

    # Check displacement source flag
    assert (df_cm["displacement_source"] == "unhcr_annual_proxy").all()
    print("  ✅ Displacement source flag: unhcr_annual_proxy")

    # Countries covered
    print(f"  ✅ Countries in output: {df_cm['ISO3'].nunique()}")

    # Flag missing Tier 1 countries
    expected_isos = TIER1_ISOS
    actual_isos = set(df_cm["ISO3"].unique())
    missing = expected_isos - actual_isos
    if missing:
        print(f"  ⚠️  Missing from output: {missing}")
        print(f"     (No UNHCR records found — will be zero-filled in master grid)")

    print("────────────────────────────────────────────\n")


def main():
    print("\n🔄 UNHCR Cleaning Pipeline")
    print("=" * 45)

    print("\n[1/6] Loading raw files...")
    df_in_raw = load_raw(INFLOWS_PATH, "Inflows")
    df_out_raw = load_raw(OUTFLOWS_PATH, "Outflows")

    print("\n[2/6] Cleaning inflows...")
    df_in = clean_inflows(df_in_raw)

    print("\n[3/6] Cleaning outflows...")
    df_out = clean_outflows(df_out_raw)

    print("\n[4/6] Aggregating to annual country stocks...")
    df_annual = aggregate_annual_stocks(df_in, df_out)
    print(f"  Annual country records: {len(df_annual)}")
    print(f"  Countries: {df_annual['ISO3'].nunique()}")

    print("\n[5/6] Expanding to country-month...")
    df_cm = expand_to_country_month(df_annual)
    print(f"  Country-month records: {len(df_cm)}")

    validate(df_in, df_out, df_cm)

    print("\n[6/6] Saving outputs...")
    df_in.to_csv(f"{INTERIM_PATH}unhcr_clean_inflows.csv", index=False)
    df_out.to_csv(f"{INTERIM_PATH}unhcr_clean_outflows.csv", index=False)
    df_cm.to_csv(f"{INTERIM_PATH}unhcr_country_month.csv", index=False)
    

    print("\n── Output Summary ──────────────────────────")
    print(f"  unhcr_clean_inflows.csv  : {len(df_in)} rows")
    print(f"  unhcr_clean_outflows.csv : {len(df_out)} rows")
    print(f"  unhcr_country_month.csv  : {len(df_cm)} country-month records")
    print(f"  Countries covered        : {df_cm['ISO3'].nunique()}")
    print(f"  Displacement source flag : unhcr_annual_proxy")
    print("────────────────────────────────────────────\n")

    return df_in, df_out, df_cm


if __name__ == "__main__":
    df_in, df_out, df_cm = main()


🔄 UNHCR Cleaning Pipeline

[1/6] Loading raw files...
  Inflows: 1554 rows × 12 columns
  Outflows: 2580 rows × 12 columns

[2/6] Cleaning inflows...
  Inflows after filter: 1554 rows
  Asylum countries: 18

[3/6] Cleaning outflows...
  Outflows after filter: 2580 rows
  Origin countries: 19

[4/6] Aggregating to annual country stocks...
  Annual country records: 110
  Countries: 19

[5/6] Expanding to country-month...
  Country-month records: 1320

── Validation ──────────────────────────────
  ✅ Year range: 2019–2024
  ⚠️  Countries not at 72 months: {'PRI': 24}
  ✅ Top inflow origins: ['Nicaragua', 'Venezuela', 'Colombia']
  ✅ Displacement source flag: unhcr_annual_proxy
  ✅ Countries in output: 19
────────────────────────────────────────────


[6/6] Saving outputs...

── Output Summary ──────────────────────────
  unhcr_clean_inflows.csv  : 1554 rows
  unhcr_clean_outflows.csv : 2580 rows
  unhcr_country_month.csv  : 1320 country-month records
  Countries covered        : 19
  Dis